# SLAM Practical Session: Using Slam Toolbox by demonstration

## Percepción en Automática y Robótica
### Master Universitario en Ingeniería Electrónica, Robótica y Automática
### Universidad de Sevilla

David Alejo Teissière

## Introduction

In this practical session we will further explore the SLAM techniques by configuring and investigating the Slam Toolbox package.

To sum up, in this lesson we will learn to:

* Use Slam Toolbox in a Turtlebot 3 simulation
* Configure the Slam Toolbox for perforing SLAM
    - For SLAM
    - For Localization
* Configure the SLAM algorithm to change its behavior
    - Turning on/off loop closing
    - Changing maximum distance sensors
    - And many more

Let's go!

## Introduction to Slam Toolbox package

Again, the main goal of this session is to autonomously generate a map of the environment. As in the previous practical session, this map will be stored as an Occupancy Grid (see Fig. 1). 

<figure style="text-align:center">
  <img src="images/inflated_maze.png" alt="" width=700>
  <figcaption>Fig. 1: Geometry map of the Turlebot3 Maze environment.  </figcaption>
</figure>

However, we will investigate the SLAM Toolbox to see which SLAM algorithms are implemented, and dive deeper in the available Front-end and Back-end algorithms


## Overview of the SLAM Toolbox

Slam Toolbox is a set of tools and capabilities for 2D SLAM built by Steve Macenski while at Simbe Robotics, maintained while at Samsung Research, and largely in his free time. Among the functionality offered by it we can highlight:

* Ordinary point-and-shoot 2D SLAM mobile robotics folks expect (start, map, save pgm file) with some nice built in utilities like saving maps

* Continuing to refine, remap, or continue mapping a saved (serialized) pose-graph at any time

* Life-long mapping: load a saved pose-graph continue mapping in a space while also removing extraneous information from newly added scans

* An optimization-based localization mode built on the pose-graph. Optionally run localization mode without a prior map for “lidar odometry” mode with local loop closures

* Synchronous and asynchronous modes of mapping

* Kinematic map merging (with an elastic graph manipulation merging technique in the works)

* Plugin-based optimization solvers with a new optimized Google Ceres based plugin

* RVIZ plugin for interacting with the tools

* Graph manipulation tools in RVIZ to manipulate nodes and connections during mapping

* Map serialization and lossless data storage

For running on live production robots, it is recommended the use of the snap: slam-toolbox, it has optimizations in it that make it about 10x faster. You need the deb/source install for the other developer level tools that don’t need to be on the robot (rviz plugins, etc).

### Cases of success

This package has been benchmarked mapping building at 5x+ real-time up to about 30,000 sq. ft. and 3x real-time up to about 60,000 sq. ft. with the largest area (I’m aware of) used was a 200,000 sq. ft. building in synchronous mode (i.e. processing all scans, regardless of lag), and much larger spaces in asynchronous mode.



### Launching the simulation environment

Download the tutorial_pkg ROS2 package to your ROS2 workspace. Then build it:

```
(docker) > cd ros2_ws/src
(docker) > git clone https://github.com/david-alejo/tutorial_pkg/
(docker) /ros2_ws> colcon build --symlink-install
```

In this case we will start a simulation of the Turtlebot 3 as indicated in the Preliminary docker ROS sesion:

```
ros2 launch turtlebot3_gazebo turtlebot3_house.launch.py
```

Then teleoperate:
```
> docker exec -it percepcion_ar bash                  (or rssa, depending on your Docker name)
(docker) > ros2 run turtlebot3_teleop turtlebot_keyboard

```

### Launching the SLAM toolbox with a YAML configuration file

From another docker terminal:

> docker exec -it percepcion_ar bash                  (or rssa, depending on your Docker name)
> ros2 launch slam_toolbox_example slam.launch.py use_sim_time:=true


In [ ]:
###The config file is as follows:

slam:
  ros__parameters:
    # Plugin params
    solver_plugin: solver_plugins::CeresSolver
    ceres_linear_solver: SPARSE_NORMAL_CHOLESKY
    ceres_preconditioner: SCHUR_JACOBI
    ceres_trust_strategy: LEVENBERG_MARQUARDT
    ceres_dogleg_type: TRADITIONAL_DOGLEG
    ceres_loss_function: None

    # ROS Parameters
    odom_frame: odom
    map_frame: map
    base_frame: base_link
    scan_topic: /scan_filtered #/scan
    mode: mapping #localization

    # if you'd like to immediately start continuing a map at a given pose
    # or at the dock, but they are mutually exclusive, if pose is given
    # will use pose
    # scan_queue_size: 1
    # map_file_name: /home/aayli/Husarion/ros2_ws/src/tutorial_pkg/maps/map_serialized
    # map_start_pose: [0.0, 0.0, 0.0]
    # map_start_at_dock: true
    debug_logging: true

    throttle_scans: 1
    transform_publish_period: 0.02 # If 0 never publishes odometry
    map_update_interval: 2.0
    resolution: 0.04
    max_laser_range: 12.0
    minimum_time_interval: 0.1
    transform_timeout: 0.2
    tf_buffer_duration: 20.0
    stack_size_to_use: 40000000
    enable_interactive_mode: false

    # General Parameters
    use_scan_matching: true
    use_scan_barycenter: true
    minimum_travel_distance: 0.3
    minimum_travel_heading: 0.5
    scan_buffer_size: 10
    scan_buffer_maximum_scan_distance: 7.0
    link_match_minimum_response_fine: 0.1
    link_scan_maximum_distance: 1.0
    loop_search_maximum_distance: 3.0
    do_loop_closing: true
    loop_match_minimum_chain_size: 10
    loop_match_maximum_variance_coarse: 3.0
    loop_match_minimum_response_coarse: 0.35
    loop_match_minimum_response_fine: 0.45

    # Correlation Parameters - Correlation Parameters
    correlation_search_space_dimension: 0.5
    correlation_search_space_resolution: 0.01
    correlation_search_space_smear_deviation: 0.1

    # Correlation Parameters - Loop Closure Parameters
    loop_search_space_dimension: 8.0
    loop_search_space_resolution: 0.05
    loop_search_space_smear_deviation: 0.03

    # Scan Matcher Parameters
    distance_variance_penalty: 0.5
    angle_variance_penalty: 1.0
    fine_search_angle_offset: 0.00349
    coarse_search_angle_offset: 0.349
    coarse_angle_resolution: 0.0349
    minimum_angle_penalty: 0.9
    minimum_distance_penalty: 0.5
    use_response_expansion: true

### Final remarks

With this part we complete the autonomous systems SLAM toolbox. 

For further SLAM exploration, you can go to the documentation of the ROS2 Slam Toolbox, which has several tutorials to generate a custom SLAM node. (https://docs.ros.org/en/jazzy/p/slam_toolbox/)

There is an interesting tutorial on multi-robot SLAM which can be executed here: (https://github.com/SteveMacenski/slam_toolbox/blob/ros2/docs/decentralized_multi_robot_slam.md)

There is an interesting example of SLAM with SlideSLAM, which has some demos with real data stored in bag files (https://github.com/KumarRobotics/SLIDE_SLAM/)

Last, PySLAM is a complete Visual SLAM framework in Python that can run examples from real datasets (https://github.com/luigifreda/pyslam)
